notes shall go here

In [12]:
import pandas as pd
import numpy as np
import plotnine as p9
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats

In [13]:
df1 = pd.read_csv("addhealth_ds3_wave1.csv")
df1.shape

(6504, 440)

In [14]:
df2 = pd.read_csv("addhealth_ds1_wave1.csv")
df2.shape

(6504, 2795)

In [15]:
df3 = pd.merge(df1, df2, on="AID", how="inner")
df3.shape

(6504, 3234)

In [16]:
df3.columns

Index(['Unnamed: 0_x', 'AID', 'SIZE', 'IDGX2', 'ODGX2', 'NOUTNOM', 'TAB113',
       'BCENT10X', 'REACH', 'REACH3',
       ...
       'PD4A', 'PD4B', 'PD4C', 'PD4D', 'PD4E', 'PD4F', 'PD5', 'PD5A', 'AH_PVT',
       'AH_RAW'],
      dtype='str', length=3234)

In [17]:
chosen_variables = [ "AID", "NOUTNOM", "HAVEBMF", "HAVEBFF", "BMFRECBF", "BFFRECBF", "ERNGRD", "ERNAGE", "BCENT10X", "REACH3", "ESRDEN", "S2", "S3", "H1WP9", "H1WP10", "H1WP13", "H1WP14", "H1WP11", "H1WP12", "H1WP15", "H1WP16", "H1MF8A", "H1MF9A", "H1FF8A", "H1FF9A", "S48", "S62E", "S62H", "S62I", "S62O", "S62R", "H1SU1", "H1RR1", "H1RE4", "H1EE3"]
                     
df4 = df3.loc[:, chosen_variables]
df4.shape

(6504, 35)

In [18]:
df4.columns

Index(['AID', 'NOUTNOM', 'HAVEBMF', 'HAVEBFF', 'BMFRECBF', 'BFFRECBF',
       'ERNGRD', 'ERNAGE', 'BCENT10X', 'REACH3', 'ESRDEN', 'S2', 'S3', 'H1WP9',
       'H1WP10', 'H1WP13', 'H1WP14', 'H1WP11', 'H1WP12', 'H1WP15', 'H1WP16',
       'H1MF8A', 'H1MF9A', 'H1FF8A', 'H1FF9A', 'S48', 'S62E', 'S62H', 'S62I',
       'S62O', 'S62R', 'H1SU1', 'H1RR1', 'H1RE4', 'H1EE3'],
      dtype='str')

In [19]:
chosen_variables2 = ["NOUTNOM", "HAVEBMF", "HAVEBFF", "BMFRECBF", "BFFRECBF", "ERNGRD", "ERNAGE", "BCENT10X", "REACH3", "ESRDEN", "H1SU1", "H1RR1"]

dfnetwork = df4.loc[:, chosen_variables2]

In [20]:
df5 = df4.dropna(subset=dfnetwork.columns, how="all").copy()
df5.shape

(6489, 35)

In [21]:
# df5.to_csv("addhealth4002.csv", index=False)

# Continue here

In [24]:
df5['H1WP10'].value_counts()

H1WP10
(5) (5) Very much      5497
(4) (4) Quite a bit     445
(3) (3) Somewhat        127
(2) (2) Very little      39
(1) (1) Not at all       15
Name: count, dtype: int64

In [27]:
# takes highest score of "how much do you think father/mother cares about you"
# higher score = higher verification to child identity (idchildveri)

df5["idchildveriraw"] = df5[
    ["H1WP10", "H1WP14"]
].max(axis=1)

df5["idchildveri"] = (
    df5["idchildveriraw"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

In [28]:
df5["idchildveri"].value_counts()

idchildveri
5.0    5815
4.0     412
3.0      92
2.0      33
1.0      10
Name: count, dtype: int64

In [29]:
df5['idchild'] = np.where(
    df5['idchildveri'] >= 4.0, 
    'Identity Verified', 
    'Identity Not Verified'
)

df5['idchild'].value_counts()

idchild
Identity Verified        6227
Identity Not Verified     262
Name: count, dtype: int64

In [116]:
df5["idchild_binary"] = (
   df5["idchild"].map({
    "Identity Available": 1,
    "Identity Not Available": 0
})
).astype("Int64")

df5['idchild_binary'].value_counts()

Series([], Name: count, dtype: Int64)

In [31]:
df5["S62E"].dtype

<StringDtype(na_value=nan)>

In [32]:
items = ["S62E", "S62H", "S62I", "S62O", "S62R"]

for item in items:
    df5[item + "_num"] = (
        df5[item]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

In [33]:
df5["S62E_num"].dtype

dtype('float64')

In [34]:
# idstudent = takes measurements of overall self-concept as a student
# right now the coding is (1 = strongly agree, 5 = strongly disagree)

df5["S62E_num_flip"] = 6 - df5['S62E_num']
df5["S62H_num_flip"] = 6 - df5['S62H_num']
df5["S62I_num_flip"] = 6 - df5['S62I_num']
df5["S62O_num_flip"] = 6 - df5['S62O_num']
df5["S62R_num_flip"] = 6 - df5['S62R_num']

In [35]:
df5["S62E_num"].head()

0    2.0
1    4.0
2    1.0
3    NaN
4    2.0
Name: S62E_num, dtype: float64

In [36]:
df5["S62E_num_flip"].head()

0    4.0
1    2.0
2    5.0
3    NaN
4    4.0
Name: S62E_num_flip, dtype: float64

In [37]:
# reverse coding idstudent so now higher score = greater verification

df5["idstudentnum"] = df5[
    ['S62E_num_flip', 'S62H_num_flip', 'S62I_num_flip', 'S62O_num_flip', 'S62R_num_flip',]
].mean(axis=1)

df5["idstudentnum"].dtype
df5["idstudentnum"].head()

0    4.0
1    2.8
2    5.0
3    NaN
4    4.2
Name: idstudentnum, dtype: float64

In [38]:
df5['S62E_num_flip'].value_counts()

S62E_num_flip
4.0    1483
5.0    1049
3.0     979
2.0     445
1.0     336
Name: count, dtype: int64

In [39]:
df5['idstudent'] = np.where(
    df5['idstudentnum'] >= 4, 
    'Identity Verified', 
    'Identity Not Verified'
)

df5['idstudent'].value_counts()

idstudent
Identity Not Verified    4440
Identity Verified        2049
Name: count, dtype: int64

In [40]:
df5["idstudent_binary"] = (
   df5["idstudent"].map({
    "Identity Verified": 1,
    "Identity Not Verified": 0
})
).astype("Int64")

df5['idstudent_binary'].value_counts()

idstudent_binary
0    4440
1    2049
Name: count, dtype: Int64

In [41]:
# Religion identity

df5["H1RE4"].value_counts()

H1RE4
(1) (1) Very important          2810
(2) (2) Fairly important        2218
(3) (3) Fairly unimportant       391
(4) (4) Not important at all     193
Name: count, dtype: int64

In [42]:
df5["idreligion"] = (
   df5["H1RE4"].map({
    "(1) (1) Very important": 1,
    "(2) (2) Fairly important": 1,
    "(3) (3) Fairly unimportant": 0,
    "(4) (4) Not important at all": 0,
})
).astype("Int64")

df5["idreligion"].value_counts()

idreligion
1    5028
0     584
Name: count, dtype: Int64

In [43]:
# Work identity

df5["H1EE3"].value_counts()

H1EE3
(1) (1) Yes    3686
(0) (0) No     2784
Name: count, dtype: int64

In [44]:
df5["idworker"] = (
   df5["H1EE3"].map({
    "(1) (1) Yes": 1,
    "(0) (0) No": 0,
})
).astype("Int64")

df5["idworker"].value_counts()

idworker
1    3686
0    2784
Name: count, dtype: Int64

In [45]:
df5["idworker"] = df5["idworker"].astype("float64")

In [46]:
df5["bmf_tieraw"] = df5[
    ["H1MF8A", "H1MF9A"]
].max(axis=1)

df5["bmf_tie"] = (
    df5["bmf_tieraw"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

df5["bff_tieraw"] = df5[
    ["H1FF8A", "H1FF9A"]
].max(axis=1)

df5["bff_tie"] = (
    df5["bff_tieraw"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

In [47]:
df5.columns

Index(['AID', 'NOUTNOM', 'HAVEBMF', 'HAVEBFF', 'BMFRECBF', 'BFFRECBF',
       'ERNGRD', 'ERNAGE', 'BCENT10X', 'REACH3', 'ESRDEN', 'S2', 'S3', 'H1WP9',
       'H1WP10', 'H1WP13', 'H1WP14', 'H1WP11', 'H1WP12', 'H1WP15', 'H1WP16',
       'H1MF8A', 'H1MF9A', 'H1FF8A', 'H1FF9A', 'S48', 'S62E', 'S62H', 'S62I',
       'S62O', 'S62R', 'H1SU1', 'H1RR1', 'H1RE4', 'H1EE3',
       'parent_disappointment', 'idchildveriraw', 'idchildveri', 'idchild',
       'idchild_binary', 'S62E_num', 'S62H_num', 'S62I_num', 'S62O_num',
       'S62R_num', 'S62E_num_flip', 'S62H_num_flip', 'S62I_num_flip',
       'S62O_num_flip', 'S62R_num_flip', 'idstudentnum', 'idstudent',
       'idstudent_binary', 'idreligion', 'idworker', 'bmf_tieraw', 'bmf_tie',
       'bff_tieraw', 'bff_tie'],
      dtype='str')

In [48]:
df5["HAVEBMF"].value_counts()

HAVEBMF
(1) (1) Ego nominiated a male best friend         2408
(0) (0) Ego didn't nominate a male best friend    1989
Name: count, dtype: int64

In [49]:
df5["HAVEBMFnum"] = (
   df5["HAVEBMF"].map({
    "(1) (1) Ego nominiated a male best friend": 1,
    "(0) (0) Ego didn't nominate a male best friend": 0
})
).astype("Int64")

df5["HAVEBFFnum"] = (
     df5["HAVEBFF"].map({
    "(1) (1) Ego nominiated a female best friend": 1,
    "(0) (0) Ego didn't nominate a female best friend": 0
})
).astype("Int64")

In [50]:
df5["HAVEBFFnum"].value_counts()

HAVEBFFnum
1    2660
0    1737
Name: count, dtype: Int64

In [51]:
df5['havebestfriend'] = df5[["HAVEBFFnum", "HAVEBMFnum"]].sum(axis=1, min_count=1).astype("Int64")

In [52]:
df5['havebestfriend'].value_counts()

havebestfriend
2    1771
1    1526
0    1100
Name: count, dtype: Int64

In [53]:
df5["havebestfriend"].unique().tolist()

[<NA>, 0, 1, 2]

In [60]:
chosen_variables = [ "AID", "idchild_binary", "HAVEBMFnum", "HAVEBFFnum", "BMFRECBFnum", "BFFRECBFnum", "idstudent_binary", "S2", "S3", "H1SU1", "bmf_tie", "bff_tie", "friend_tiestrength", "havebestfriend", "H1RR1", "reciprocatedbff", "NOUTNOM", "idreligion", "idworker"]
                     
df6 = df5.loc[:, chosen_variables]
df6.columns

Index(['AID', 'idchild_binary', 'HAVEBMFnum', 'HAVEBFFnum', 'BMFRECBFnum',
       'BFFRECBFnum', 'idstudent_binary', 'S2', 'S3', 'H1SU1', 'bmf_tie',
       'bff_tie', 'friend_tiestrength', 'havebestfriend', 'H1RR1',
       'reciprocatedbff', 'NOUTNOM', 'idreligion', 'idworker'],
      dtype='str')

In [64]:
df6["H1RR1"].value_counts()

H1RR1
(1) (1) Yes    3582
(0) (0) No     2878
Name: count, dtype: int64

In [65]:
df6["H1RR1"].dtype

<StringDtype(na_value=nan)>

In [66]:
df6["romanticpartner"] = (
    df6["H1RR1"].map({
    "(1) (1) Yes" : 1,
    "(0) (0) No": 0
})
)

In [67]:
df6["romanticpartner"].dtype

dtype('float64')

In [68]:
df6.head()

# best male friend (bmf) and best female friend (bff) ties: higher score = closer tie (Granovetter)
# idself con: lower = more verified

,AID,idchild_binary,HAVEBMFnum,HAVEBFFnum,BMFRECBFnum,BFFRECBFnum,idstudent_binary,S2,S3,H1SU1,...,friend_tiestrength,havebestfriend,H1RR1,reciprocatedbff,NOUTNOM,idreligion,idworker,NOUTNOM_treated,NOUTNOM_binary,romanticpartner
0,57100270,1,<NA>,<NA>,NaN,NaN,1,(2) (2) Female,12.0,(0) (0) No (skip to Q.4),...,0.25,<NA>,(1) (1) Yes,<NA>,NaN,1,0.0,Identity Not Verified,0,1.0
1,57101310,1,<NA>,<NA>,NaN,NaN,0,(2) (2) Female,11.0,(0) (0) No (skip to Q.4),...,1.00,<NA>,(1) (1) Yes,<NA>,NaN,<NA>,0.0,Identity Not Verified,0,1.0
2,57103171,1,<NA>,<NA>,NaN,NaN,1,(1) (1) Male,10.0,(0) (0) No (skip to Q.4),...,0.00,<NA>,(0) (0) No,<NA>,NaN,1,1.0,Identity Not Verified,0,0.0
3,57103869,1,<NA>,<NA>,NaN,NaN,0,NaN,NaN,(1) (1) Yes,...,0.50,<NA>,(1) (1) Yes,<NA>,NaN,<NA>,1.0,Identity Not Verified,0,1.0
4,57104553,1,<NA>,<NA>,NaN,NaN,1,(2) (2) Female,12.0,(1) (1) Yes,...,0.50,<NA>,(1) (1) Yes,<NA>,NaN,1,0.0,Identity Not Verified,0,1.0


In [69]:
df6["H1SU1"].value_counts()

H1SU1
(0) (0) No (skip to Q.4)    5614
(1) (1) Yes                  821
Name: count, dtype: int64

In [70]:
df6["H1SU1"].dtype
df6["H1SU1"].head()

0    (0) (0) No (skip to Q.4)
1    (0) (0) No (skip to Q.4)
2    (0) (0) No (skip to Q.4)
3                 (1) (1) Yes
4                 (1) (1) Yes
Name: H1SU1, dtype: str

In [71]:
df6["suicide"] = (
   df6["H1SU1"].map({
    "(1) (1) Yes": 1,
    "(0) (0) No (skip to Q.4)": 0
})
)

df6["suicide"].value_counts()

suicide
0.0    5614
1.0     821
Name: count, dtype: int64

# Models

In [ ]:

from scipy.stats import chi2
from stargazer.stargazer import Stargazer
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

In [73]:
from marginaleffects import avg_comparisons, comparisons

# Getting descriptive stats

In [74]:
df6.shape

(6489, 23)

In [75]:
df6.columns

Index(['AID', 'idchild_binary', 'HAVEBMFnum', 'HAVEBFFnum', 'BMFRECBFnum',
       'BFFRECBFnum', 'idstudent_binary', 'S2', 'S3', 'H1SU1', 'bmf_tie',
       'bff_tie', 'friend_tiestrength', 'havebestfriend', 'H1RR1',
       'reciprocatedbff', 'NOUTNOM', 'idreligion', 'idworker',
       'NOUTNOM_treated', 'NOUTNOM_binary', 'romanticpartner', 'suicide'],
      dtype='str')

In [76]:
chosen_variables3 = ['AID', 'idchild_binary', 'HAVEBMFnum', 'HAVEBFFnum', 'BMFRECBFnum',
       'BFFRECBFnum', 'idstudent_binary', 'S2', 'S3', 'bmf_tie',
       'bff_tie', 'friend_tiestrength', 'havebestfriend',
       'romanticpartner', 'suicide', 'reciprocatedbff', 'NOUTNOM_binary', 'NOUTNOM', 'idreligion', 'idworker']
                     
df7 = df6.loc[:, chosen_variables3]
df7.columns

Index(['AID', 'idchild_binary', 'HAVEBMFnum', 'HAVEBFFnum', 'BMFRECBFnum',
       'BFFRECBFnum', 'idstudent_binary', 'S2', 'S3', 'bmf_tie', 'bff_tie',
       'friend_tiestrength', 'havebestfriend', 'romanticpartner', 'suicide',
       'reciprocatedbff', 'NOUTNOM_binary', 'NOUTNOM', 'idreligion',
       'idworker'],
      dtype='str')

In [77]:
df7.shape

(6489, 20)

In [78]:
print(df7.isna().sum()[df7.isna().sum() > 0])

HAVEBMFnum            2092
HAVEBFFnum            2092
BMFRECBFnum           4526
BFFRECBFnum           6489
S2                    1772
S3                    1775
bmf_tie                526
bff_tie                717
friend_tiestrength     138
havebestfriend        2092
romanticpartner         29
suicide                 54
reciprocatedbff       4526
NOUTNOM               2092
idreligion             877
idworker                19
dtype: int64


In [79]:
chosen_variables4 = "idstudent_binary", "suicide", "idchild_binary", "idreligion", "idworker", "romanticpartner", "havebestfriend"

dfvars = df7.loc[:, chosen_variables4]

df8 = df7.dropna(subset=dfvars.columns, how="any").copy()
df8.shape

(3847, 20)

In [80]:
print(df8.isna().sum()[df8.isna().sum() > 0])

BMFRECBFnum           2103
BFFRECBFnum           3847
S2                      27
S3                      30
bmf_tie                290
bff_tie                405
friend_tiestrength      65
reciprocatedbff       2103
dtype: int64


In [81]:
df8['havebestfriend'] = df8['havebestfriend'].astype(float)
df8['havebestfriend'].value_counts()

havebestfriend
2.0    1545
1.0    1354
0.0     948
Name: count, dtype: int64

In [82]:
df8['romanticpartner'].value_counts()

romanticpartner
1.0    2111
0.0    1736
Name: count, dtype: int64

In [83]:
df8['idstudent_binary'].value_counts()

idstudent_binary
0    2141
1    1706
Name: count, dtype: Int64

In [84]:
df8['idchild_binary'].value_counts()

idchild_binary
1    3748
0      99
Name: count, dtype: Int64

In [85]:
df8['idreligion'].value_counts()

idreligion
1    3467
0     380
Name: count, dtype: Int64

In [86]:
df8['idworker'].value_counts()

idworker
1.0    2166
0.0    1681
Name: count, dtype: int64

# Logit 1: suicide ~ ID student

In [87]:
df8['idstudent_binary'].value_counts()

idstudent_binary
0    2141
1    1706
Name: count, dtype: Int64

In [88]:
lm1 = smf.logit(
	'suicide ~ idstudent_binary',
	data=df8
).fit()
# print(lm1.summary())

Optimization terminated successfully.
         Current function value: 0.347790
         Iterations 6


In [89]:
# lm1 = smf.logit('suicide ~ idstudent',data=df8).fit()
# print(lm1.summary())

In [90]:
lm2 = smf.logit(
	'suicide ~ idstudent_binary + idchild_binary',
	data=df8
).fit()
# print(lm2.summary())

Optimization terminated successfully.
         Current function value: 0.346209
         Iterations 7


In [92]:
lm4 = smf.logit(
	'suicide ~ idstudent_binary + idchild_binary + havebestfriend',
	data=df8
).fit()
# print(lm4.summary())

Optimization terminated successfully.
         Current function value: 0.345750
         Iterations 7


In [93]:
lm5 = smf.logit(
	'suicide ~ idstudent_binary + idchild_binary + havebestfriend + romanticpartner',
	data=df8
).fit()
# print(lm5.summary())

Optimization terminated successfully.
         Current function value: 0.342069
         Iterations 7


In [94]:
df8["idreligion"] = df8["idreligion"].astype("float64")

df8["idreligion"].value_counts()

idreligion
1.0    3467
0.0     380
Name: count, dtype: int64

In [95]:
lm6 = smf.logit(
	'suicide ~ idstudent_binary + idchild_binary + idreligion + idworker + romanticpartner + havebestfriend',
	data=df8
).fit()
print(lm6.summary())

Optimization terminated successfully.
         Current function value: 0.341060
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                suicide   No. Observations:                 3847
Model:                          Logit   Df Residuals:                     3840
Method:                           MLE   Df Model:                            6
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                 0.03936
Time:                        15:35:16   Log-Likelihood:                -1312.1
converged:                       True   LL-Null:                       -1365.8
Covariance Type:            nonrobust   LLR p-value:                 6.767e-21
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -1.1686      0.284     -4.113      0.000      -1.726      -0.612
idstudent_b

In [96]:
print(avg_comparisons(lm6))

shape: (6, 9)
┌──────────────────┬──────────┬──────────┬───────────┬───┬──────────┬──────┬──────────┬──────────┐
│ Term             ┆ Contrast ┆ Estimate ┆ Std.Error ┆ … ┆ P(>|z|)  ┆ S    ┆ 2.5%     ┆ 97.5%    │
│ ---              ┆ ---      ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---  ┆ ---      ┆ ---      │
│ str              ┆ str      ┆ str      ┆ str       ┆   ┆ str      ┆ str  ┆ str      ┆ str      │
╞══════════════════╪══════════╪══════════╪═══════════╪═══╪══════════╪══════╪══════════╪══════════╡
│ havebestfriend   ┆ +1       ┆ 0.0107   ┆ 0.0065    ┆ … ┆ 0.101    ┆ 3.31 ┆ -0.00207 ┆ 0.0234   │
│ idchild_binary   ┆ 1 - 0    ┆ -0.1     ┆ 0.0383    ┆ … ┆ 0.00901  ┆ 6.79 ┆ -0.175   ┆ -0.025   │
│ idreligion       ┆ 1 - 0    ┆ -0.0429  ┆ 0.0183    ┆ … ┆ 0.0191   ┆ 5.71 ┆ -0.0789  ┆ -0.00703 │
│ idstudent_binary ┆ 1 - 0    ┆ -0.076   ┆ 0.00995   ┆ … ┆ 2.75e-14 ┆ 45   ┆ -0.0955  ┆ -0.0565  │
│ idworker         ┆ 1 - 0    ┆ 0.0124   ┆ 0.0102    ┆ … ┆ 0.226    ┆ 2.14 ┆ -0.00767 ┆ 0.0324 

In [97]:
avg_comparisons(lm6, variables='idstudent_binary')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""idstudent_binary""","""1 - 0""",-0.076001,0.00995,-7.638315,2.7534e-14,45.045804,-0.095509,-0.056493


In [98]:
import polars as pl
from marginaleffects import avg_predictions

intercept_grid = pl.DataFrame({
    "idstudent_binary": [0.0],
    "idchild_binary": [0.0],
    "idreligion": [0.0],
    "idworker": [0.0],
    "romanticpartner": [0.0],
    "havebestfriend": [0.0],
})

interceptlm6 = avg_predictions(lm6, newdata=intercept_grid)

print(interceptlm6)

shape: (1, 7)
┌──────────┬───────────┬──────┬──────────┬──────┬───────┬───────┐
│ Estimate ┆ Std.Error ┆ z    ┆ P(>|z|)  ┆ S    ┆ 2.5%  ┆ 97.5% │
│ ---      ┆ ---       ┆ ---  ┆ ---      ┆ ---  ┆ ---   ┆ ---   │
│ str      ┆ str       ┆ str  ┆ str      ┆ str  ┆ str   ┆ str   │
╞══════════╪═══════════╪══════╪══════════╪══════╪═══════╪═══════╡
│ 0.237    ┆ 0.0514    ┆ 4.61 ┆ 4.09e-06 ┆ 17.9 ┆ 0.136 ┆ 0.338 │
└──────────┴───────────┴──────┴──────────┴──────┴───────┴───────┘


# Stargazer

In [99]:
sg1 = Stargazer([lm1, lm2, lm3])
sg1.title('Effects Suicide Ideation')
sg1.dependent_variable_name('Suicide Ideation')

sg1.custom_columns(['Model 1', 'Model 2', 'Model 3'], [1, 1, 1])
sg1.covariate_order([
	'idstudent_binary',
	'idchild_binary',
	'friend_tiestrength',
	'Intercept'
])
sg1.rename_covariates({
	'idstudent_binary':             'Student identity verified',
	'idchild_binary':               'Child identity verified',
	'friend_tiestrength':           'Strength of best friend ties',
	'Intercept':                    'Intercept'
})

# sg1

In [100]:
sg2 = Stargazer([lm1, lm2, lm4, lm5, lm6])
sg2.title('Effects on Suicide Ideation')
sg2.dependent_variable_name('Suicide Ideation')

sg2.custom_columns(['Model 1', 'Model 2', 'Model 4', 'Model 5', 'Model 6'], [1, 1, 1, 1, 1])
sg2.covariate_order([
	'idstudent_binary',
	'idchild_binary',
	'idreligion',
	'idworker',
	'havebestfriend',
	'romanticpartner',
	'Intercept'
])

sg2.rename_covariates({
	'idstudent_binary':             'Student identity verified',
	'idchild_binary':               'Child identity',
	'idreligion':					'Religious identity',
	'idworker':						'Vocational identity',
	'havebestfriend':      	        'Peer identity',
	'romanticpartner':				'Romantic partner identity',
	'Intercept':                    'Intercept'
})

# sg2

# Heterogeneity

In [101]:
df8[['idstudent_binary', 'idchild_binary', 'idreligion', 'idworker', 'havebestfriend', 'romanticpartner']].head()

,idstudent_binary,idchild_binary,idreligion,idworker,havebestfriend,romanticpartner
173,0,1,0.0,0.0,0.0,1.0
174,1,1,1.0,0.0,1.0,0.0
175,1,1,1.0,1.0,1.0,1.0
177,0,1,1.0,0.0,2.0,0.0
178,0,1,1.0,1.0,1.0,1.0


In [102]:
df8["havebestfriend"].value_counts()

havebestfriend
2.0    1545
1.0    1354
0.0     948
Name: count, dtype: int64

In [103]:
df8["idfriend_binary"] = (
   df8["havebestfriend"].map({
    2.0: 1,
    1.0: 1,
    0.0: 0
})
).astype("float")

df8['idfriend_binary'].value_counts()

idfriend_binary
1.0    2899
0.0     948
Name: count, dtype: int64

In [104]:
df8[['idstudent_binary', 'idchild_binary', 'idreligion', 'idworker', 'idfriend_binary', 'romanticpartner']].head()

,idstudent_binary,idchild_binary,idreligion,idworker,idfriend_binary,romanticpartner
173,0,1,0.0,0.0,0.0,1.0
174,1,1,1.0,0.0,1.0,0.0
175,1,1,1.0,1.0,1.0,1.0
177,0,1,1.0,0.0,1.0,0.0
178,0,1,1.0,1.0,1.0,1.0


In [105]:
df8["heterogeneity"] = df8[
    ['idchild_binary', 'idreligion', 'idworker', 'idfriend_binary', 'romanticpartner']
].sum(axis=1)

df8["heterogeneity"].value_counts()

heterogeneity
4.0    1458
3.0    1137
5.0     892
2.0     329
1.0      30
0.0       1
Name: count, dtype: Int64

In [106]:
lm7 = smf.logit(
	'suicide ~ idstudent_binary + idchild_binary + idreligion + idworker + romanticpartner + havebestfriend + heterogeneity',
	data=df8
).fit()
print(lm7.summary())

Optimization terminated successfully.
         Current function value: 0.340909
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                suicide   No. Observations:                 3847
Model:                          Logit   Df Residuals:                     3839
Method:                           MLE   Df Model:                            7
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                 0.03978
Time:                        15:35:16   Log-Likelihood:                -1311.5
converged:                       True   LL-Null:                       -1365.8
Covariance Type:            nonrobust   LLR p-value:                 1.732e-20
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -1.1229      0.287     -3.912      0.000      -1.685      -0.560
idstudent_b

In [107]:
lm8 = smf.logit(
	'suicide ~ idstudent_binary + idchild_binary + idreligion + idworker + romanticpartner + havebestfriend + heterogeneity + idstudent_binary:heterogeneity',
	data=df8
).fit()
print(lm8.summary())

Optimization terminated successfully.
         Current function value: 0.340039
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                suicide   No. Observations:                 3847
Model:                          Logit   Df Residuals:                     3838
Method:                           MLE   Df Model:                            8
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                 0.04223
Time:                        15:35:16   Log-Likelihood:                -1308.1
converged:                       True   LL-Null:                       -1365.8
Covariance Type:            nonrobust   LLR p-value:                 3.001e-21
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -1.4235      0.313     -4.552     

In [108]:
print(avg_comparisons(lm7))

shape: (7, 9)
┌──────────────────┬──────────┬──────────┬───────────┬───┬──────────┬───────┬─────────┬─────────┐
│ Term             ┆ Contrast ┆ Estimate ┆ Std.Error ┆ … ┆ P(>|z|)  ┆ S     ┆ 2.5%    ┆ 97.5%   │
│ ---              ┆ ---      ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---   ┆ ---     ┆ ---     │
│ str              ┆ str      ┆ str      ┆ str       ┆   ┆ str      ┆ str   ┆ str     ┆ str     │
╞══════════════════╪══════════╪══════════╪═══════════╪═══╪══════════╪═══════╪═════════╪═════════╡
│ havebestfriend   ┆ +1       ┆ 0.0212   ┆ 0.0118    ┆ … ┆ 0.072    ┆ 3.8   ┆ -0.0019 ┆ 0.0443  │
│ heterogeneity    ┆ +1       ┆ -0.0233  ┆ 0.0218    ┆ … ┆ 0.284    ┆ 1.81  ┆ -0.0661 ┆ 0.0194  │
│ idchild_binary   ┆ 1 - 0    ┆ -0.0639  ┆ 0.046     ┆ … ┆ 0.164    ┆ 2.6   ┆ -0.154  ┆ 0.0262  │
│ idreligion       ┆ 1 - 0    ┆ -0.0157  ┆ 0.0287    ┆ … ┆ 0.584    ┆ 0.777 ┆ -0.0719 ┆ 0.0405  │
│ idstudent_binary ┆ 1 - 0    ┆ -0.0759  ┆ 0.00995   ┆ … ┆ 3.06e-14 ┆ 44.9  ┆ -0.0954 ┆ -0.0564 │
│ idwo

In [109]:
intercept_grid = pl.DataFrame({
    "idstudent_binary": [0.0],
    "idchild_binary": [0.0],
    "idreligion": [0.0],
    "idworker": [0.0],
    "romanticpartner": [0.0],
    "havebestfriend": [0.0],
    "heterogeneity": [0.0]
})

interceptlm7 = avg_predictions(lm7, newdata=intercept_grid)

print(interceptlm7)

shape: (1, 7)
┌──────────┬───────────┬──────┬──────────┬──────┬───────┬───────┐
│ Estimate ┆ Std.Error ┆ z    ┆ P(>|z|)  ┆ S    ┆ 2.5%  ┆ 97.5% │
│ ---      ┆ ---       ┆ ---  ┆ ---      ┆ ---  ┆ ---   ┆ ---   │
│ str      ┆ str       ┆ str  ┆ str      ┆ str  ┆ str   ┆ str   │
╞══════════╪═══════════╪══════╪══════════╪══════╪═══════╪═══════╡
│ 0.245    ┆ 0.0532    ┆ 4.62 ┆ 4.01e-06 ┆ 17.9 ┆ 0.141 ┆ 0.35  │
└──────────┴───────────┴──────┴──────────┴──────┴───────┴───────┘


In [110]:
print(avg_comparisons(lm8))

shape: (7, 9)
┌──────────────────┬──────────┬──────────┬───────────┬───┬──────────┬───────┬───────────┬─────────┐
│ Term             ┆ Contrast ┆ Estimate ┆ Std.Error ┆ … ┆ P(>|z|)  ┆ S     ┆ 2.5%      ┆ 97.5%   │
│ ---              ┆ ---      ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---   ┆ ---       ┆ ---     │
│ str              ┆ str      ┆ str      ┆ str       ┆   ┆ str      ┆ str   ┆ str       ┆ str     │
╞══════════════════╪══════════╪══════════╪═══════════╪═══╪══════════╪═══════╪═══════════╪═════════╡
│ havebestfriend   ┆ +1       ┆ 0.0225   ┆ 0.0118    ┆ … ┆ 0.0572   ┆ 4.13  ┆ -0.000684 ┆ 0.0456  │
│ heterogeneity    ┆ +1       ┆ -0.0275  ┆ 0.0219    ┆ … ┆ 0.21     ┆ 2.25  ┆ -0.0704   ┆ 0.0155  │
│ idchild_binary   ┆ 1 - 0    ┆ -0.0619  ┆ 0.0453    ┆ … ┆ 0.172    ┆ 2.54  ┆ -0.151    ┆ 0.0269  │
│ idreligion       ┆ 1 - 0    ┆ -0.0123  ┆ 0.0281    ┆ … ┆ 0.661    ┆ 0.598 ┆ -0.0675   ┆ 0.0428  │
│ idstudent_binary ┆ 1 - 0    ┆ -0.0747  ┆ 0.0101    ┆ … ┆ 1.65e-13 ┆ 42.5  ┆ -0.0945 

In [111]:
intercept_grid = pl.DataFrame({
    "idstudent_binary": [0.0],
    "idchild_binary": [0.0],
    "idreligion": [0.0],
    "idworker": [0.0],
    "romanticpartner": [0.0],
    "havebestfriend": [0.0],
    "heterogeneity": [0.0]
})

interceptlm8 = avg_predictions(lm8, newdata=intercept_grid)

print(interceptlm8)

shape: (1, 7)
┌──────────┬───────────┬──────┬──────────┬──────┬────────┬───────┐
│ Estimate ┆ Std.Error ┆ z    ┆ P(>|z|)  ┆ S    ┆ 2.5%   ┆ 97.5% │
│ ---      ┆ ---       ┆ ---  ┆ ---      ┆ ---  ┆ ---    ┆ ---   │
│ str      ┆ str       ┆ str  ┆ str      ┆ str  ┆ str    ┆ str   │
╞══════════╪═══════════╪══════╪══════════╪══════╪════════╪═══════╡
│ 0.194    ┆ 0.0489    ┆ 3.97 ┆ 7.39e-05 ┆ 13.7 ┆ 0.0982 ┆ 0.29  │
└──────────┴───────────┴──────┴──────────┴──────┴────────┴───────┘


In [112]:
import plotnine as p9
from marginaleffects import predictions

pred_df8 = predictions(lm8, by=["heterogeneity", "idstudent_binary"]).to_pandas()

plot = (
    p9.ggplot(pred_df8, p9.aes(x='heterogeneity', y='estimate', color='factor(idstudent_binary)')) +
    p9.geom_line(size=1) +
    p9.geom_point() +
    p9.theme_light() +
    p9.theme_minimal() + 
    p9.scale_color_manual(values=['#cb4335', '#2e4053'], labels=['Student ID not verified', 'Student ID verified']) +
    p9.scale_fill_manual(values=['#cb4335', '#2e4053'], labels=['Student ID not verified', 'Student ID verified']) +
    p9.labs(x='Role Heterogeneity', y='Suicide ideation',
            title='Effect of role heterogeneity comparing student identity verification',
            color='Student Identity Verification')
)

# plot.save('interaction_plot2.png', width=25, height=10.5, units='cm')

In [113]:
from marginaleffects import avg_slopes

interaction_effect = avg_slopes(
    lm8, 
    variables="heterogeneity",   
    by="idstudent_binary"
)

print(interaction_effect)

shape: (2, 8)
┌──────────────────┬──────────┬───────────┬────────┬─────────┬──────┬─────────┬──────────┐
│ idstudent_binary ┆ Estimate ┆ Std.Error ┆ z      ┆ P(>|z|) ┆ S    ┆ 2.5%    ┆ 97.5%    │
│ ---              ┆ ---      ┆ ---       ┆ ---    ┆ ---     ┆ ---  ┆ ---     ┆ ---      │
│ str              ┆ str      ┆ str       ┆ str    ┆ str     ┆ str  ┆ str     ┆ str      │
╞══════════════════╪══════════╪═══════════╪════════╪═════════╪══════╪═════════╪══════════╡
│ 0                ┆ -0.0227  ┆ 0.0274    ┆ -0.829 ┆ 0.407   ┆ 1.3  ┆ -0.0766 ┆ 0.0311   │
│ 1                ┆ -0.0332  ┆ 0.0163    ┆ -2.04  ┆ 0.0419  ┆ 4.58 ┆ -0.0651 ┆ -0.00122 │
└──────────────────┴──────────┴───────────┴────────┴─────────┴──────┴─────────┴──────────┘
Term: heterogeneity
Contrast: dY/dX


In [114]:
sg3 = Stargazer([lm6, lm7, lm8])
sg3.title('Effects on Suicide Ideation')
sg3.dependent_variable_name('Suicide Ideation')

sg3.custom_columns(['Independent Variables', 'Includes Heterogeneity', 'Interaction Effect'], [1, 1, 1])
sg3.covariate_order([
	'idstudent_binary',
	'idchild_binary',
	'idreligion',
	'idworker',
	'havebestfriend',
	'romanticpartner',
    'heterogeneity',
	'idstudent_binary:heterogeneity',
	'Intercept'
])

sg3.rename_covariates({
	'idstudent_binary':               'Student identity verified',
	'idchild_binary':                 'Child identity',
	'idreligion':					  'Religious identity',
	'idworker':						  'Vocational identity',
	'havebestfriend':      	          'Peer identity',
	'romanticpartner':				  'Romantic partner identity',
    'heterogeneity':                  'Availability of alternative ties',
    'idstudent_binary:heterogeneity': 'Interaction',
	'Intercept':                      'Intercept'
})

sg3

In [115]:
# Overleaf

sg3 = Stargazer([lm6, lm7, lm8])
sg3.title('Effects on Suicide Ideation')
sg3.dependent_variable_name('Suicide Ideation')

sg3.custom_columns(['Model 1', 'Model 2', 'With Interaction'], [1, 1, 1])
sg3.covariate_order([
	'idstudent_binary',
	'idchild_binary',
	'idreligion',
	'idworker',
	'havebestfriend',
	'romanticpartner',
    'heterogeneity',
	'idstudent_binary:heterogeneity',
	'Intercept'
])

sg3.rename_covariates({
	'idstudent_binary':               'Student identity verified',
	'idchild_binary':                 'Child identity',
	'idreligion':					  'Religious identity',
	'idworker':						  'Vocational identity',
	'havebestfriend':      	          'Best friend identity',
	'romanticpartner':				  'Romantic partner identity',
    'heterogeneity':                  'Alternative ties',
    'idstudent_binary:heterogeneity': 'Interaction',
	'Intercept':                      'Intercept'
})

print(sg3.render_latex())

\begin{table}[!htbp] \centering
  \caption{Effects on Suicide Ideation}
\begin{tabular}{@{\extracolsep{5pt}}lccc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{3}{c}{\textit{Dependent variable: Suicide Ideation}} \
\cr \cline{2-4}
\\[-1.8ex] & \multicolumn{1}{c}{Model 1} & \multicolumn{1}{c}{Model 2} & \multicolumn{1}{c}{With Interaction}  \\
\\[-1.8ex] & (1) & (2) & (3) \\
\hline \\[-1.8ex]
 Student identity verified & -0.817$^{***}$ & -0.816$^{***}$ & 0.407$^{}$ \\
& (0.114) & (0.114) & (0.478) \\
 Child identity & -0.787$^{***}$ & -0.543$^{}$ & -0.530$^{}$ \\
& (0.243) & (0.332) & (0.331) \\
 Religious identity & -0.392$^{***}$ & -0.154$^{}$ & -0.122$^{}$ \\
& (0.151) & (0.268) & (0.269) \\
 Vocational identity & 0.127$^{}$ & 0.366$^{}$ & 0.408$^{*}$ \\
& (0.106) & (0.246) & (0.247) \\
 Best friend identity & 0.109$^{}$ & 0.216$^{*}$ & 0.229$^{*}$ \\
& (0.066) & (0.120) & (0.120) \\
 Romantic partner identity & 0.538$^{***}$ & 0.778$^{***}$ & 0.812$^{***}$ \\
& (0.109) & (0.249)